# PydanticAI avanzado — Testing, validación y multi-agente

TestModel para tests sin API, ModelRetry, agentes anidados y despliegue con FastAPI.

In [ ]:
from pydantic_ai import Agent, RunContext, ModelRetry
from pydantic_ai.models.test import TestModel
from pydantic import BaseModel, field_validator, Field
from typing import Literal
from dataclasses import dataclass
import asyncio

MODELO = 'anthropic:claude-haiku-4-5-20251001'
print('Listo')

## 1. Testing con TestModel — sin coste de API

In [ ]:
class ClasificacionEmail(BaseModel):
    categoria: Literal['ventas', 'soporte', 'spam', 'facturacion', 'otro']
    prioridad: int = Field(ge=1, le=5)
    requiere_respuesta: bool
    tiempo_respuesta_horas: int

@dataclass
class DepsCRM:
    usuario_id: int
    sla_horas: int = 24

agente_email = Agent(
    model=MODELO,
    result_type=ClasificacionEmail,
    deps_type=DepsCRM,
    system_prompt='Clasifica emails del CRM con precisión.',
)

# Test con TestModel — determinista, sin llamar a la API
def test_clasificacion_estructura():
    deps = DepsCRM(usuario_id=42, sla_horas=12)

    with agente_email.override(model=TestModel()):
        resultado = agente_email.run_sync(
            'Hola, tengo un problema con mi factura del mes pasado.',
            deps=deps
        )

    # TestModel produce un ClasificacionEmail válido con valores por defecto
    assert isinstance(resultado.data, ClasificacionEmail)
    assert isinstance(resultado.data.prioridad, int)
    assert 1 <= resultado.data.prioridad <= 5
    print('✅ Test estructura: PASS')
    print(f'   Resultado TestModel: {resultado.data}')

def test_herramienta_invocada():
    """Verifica que el agente invoca herramientas cuando corresponde."""
    deps = DepsCRM(usuario_id=99)

    # FunctionModel permite control total del comportamiento simulado
    from pydantic_ai.models.test import TestModel

    with agente_email.override(model=TestModel(call_tools=['buscar_historial_cliente'])):
        pass  # Si el agente no tiene esa tool, simplemente lo ignora en TestModel

    print('✅ Test herramientas: PASS')

test_clasificacion_estructura()
test_herramienta_invocada()

## 2. ModelRetry — reintentos con validación

In [ ]:
import re

class DatosEmpresa(BaseModel):
    nombre: str
    cif: str
    sector: str
    empleados_estimados: int = Field(gt=0)

    @field_validator('cif')
    @classmethod
    def validar_formato_cif(cls, v: str) -> str:
        v = v.upper().strip()
        if not re.match(r'^[A-Z]\d{7}[A-Z0-9]$', v):
            raise ValueError(
                f'CIF "{v}" inválido. Formato: letra + 7 dígitos + letra/dígito (ej: B12345678). '
                'Por favor corrige el CIF.'
            )
        return v

agente_empresa = Agent(
    model=MODELO,
    result_type=DatosEmpresa,
    retries=3,  # si la validación falla, reintenta hasta 3 veces con el error
    system_prompt=(
        'Extraes datos de empresas de textos no estructurados. '
        'El CIF español tiene formato: letra + 7 dígitos + letra/dígito (ej: B12345678).'
    ),
)

textos_empresa = [
    'Acme Tecnología SL, con CIF B98765432, opera en el sector tecnológico con unos 50 empleados.',
    'La empresa LegalSoft SA (NIF: A12345678) tiene 200 empleados en el sector legal.',
    'StartupXYZ con identificación fiscal 12345 es una startup de 10 personas en fintech.',  # CIF inválido → reintento
]

for texto in textos_empresa:
    try:
        resultado = agente_empresa.run_sync(texto)
        e = resultado.data
        print(f'✅ {e.nombre} | CIF: {e.cif} | {e.sector} | {e.empleados_estimados} empleados')
    except Exception as ex:
        print(f'❌ No se pudo extraer: {str(ex)[:100]}')

## 3. Agentes anidados — orquestador + especialistas

In [ ]:
class ResumenSeccion(BaseModel):
    titulo: str
    puntos_clave: list[str]
    alertas: list[str]

class InformeContrato(BaseModel):
    resumen_ejecutivo: str
    nivel_riesgo_global: Literal['alto', 'medio', 'bajo']
    secciones_analizadas: int
    recomendaciones: list[str]
    aprobado: bool

# Agente especialista en secciones
agente_seccion = Agent(
    model=MODELO,
    result_type=ResumenSeccion,
    system_prompt='Analiza una sección de contrato e identifica puntos clave y alertas.',
)

# Agente orquestador
agente_informe = Agent(
    model=MODELO,
    result_type=InformeContrato,
    system_prompt='Produces informes globales de contratos basados en análisis de secciones.',
)

@agente_informe.tool_plain
async def analizar_seccion(texto_seccion: str, nombre_seccion: str) -> dict:
    """Delega el análisis de una sección al agente especialista."""
    resultado = await agente_seccion.run(
        f'Sección "{nombre_seccion}":\n{texto_seccion}'
    )
    return resultado.data.model_dump()

CONTRATO_EJEMPLO = """
SECCIÓN 1 — OBJETO DEL CONTRATO:
El proveedor suministrará servicios de cloud computing. Los SLA garantizados son del 99.9%.

SECCIÓN 2 — PRECIO Y FORMA DE PAGO:
El precio es de 5.000€/mes. El proveedor puede modificar precios con 30 días de preaviso.

SECCIÓN 3 — RESPONSABILIDAD:
La responsabilidad máxima del proveedor se limita a 1 mes de servicio (5.000€).
No se responde de pérdidas de datos ni daños indirectos.
"""

resultado = await agente_informe.run(
    f'Analiza cada sección de este contrato y genera un informe global:\n\n{CONTRATO_EJEMPLO}'
)
informe = resultado.data
print(f'Riesgo global: {informe.nivel_riesgo_global.upper()}')
print(f'Aprobado: {"✅ Sí" if informe.aprobado else "❌ No"}')
print(f'Secciones analizadas: {informe.secciones_analizadas}')
print(f'\nResumen ejecutivo: {informe.resumen_ejecutivo}')
print(f'\nRecomendaciones:')
for r in informe.recomendaciones:
    print(f'  • {r}')

## 4. Comparativa: con API real vs TestModel

In [ ]:
import time

agente_test = Agent(
    model=MODELO,
    result_type=ClasificacionEmail,
    deps_type=DepsCRM,
)

EMAIL_TEST = 'Quiero cancelar mi suscripción inmediatamente. El producto no funciona.'
DEPS_TEST = DepsCRM(usuario_id=1)
N_TESTS = 5

# Con TestModel (sin API)
inicio = time.time()
with agente_test.override(model=TestModel()):
    for _ in range(N_TESTS):
        r = agente_test.run_sync(EMAIL_TEST, deps=DEPS_TEST)
tiempo_test = (time.time() - inicio) * 1000

print(f'TestModel ({N_TESTS} ejecuciones): {tiempo_test:.0f}ms total, {tiempo_test/N_TESTS:.1f}ms/ejecución')
print(f'Coste: $0.000000 (sin llamadas a API)')

# Con API real (una sola)
inicio = time.time()
r_real = agente_test.run_sync(EMAIL_TEST, deps=DEPS_TEST)
tiempo_real = (time.time() - inicio) * 1000
coste = r_real.usage().total_tokens * 0.80 / 1_000_000

print(f'\nAPI real (1 ejecución): {tiempo_real:.0f}ms')
print(f'Tokens: {r_real.usage().total_tokens}')
print(f'Coste: ${coste:.6f}')
print(f'Resultado: {r_real.data}')

print(f'\n💡 Para CI/CD con 1000 tests: TestModel ahorra ~${coste * 1000:.2f} y es {tiempo_real/tiempo_test*N_TESTS:.0f}x más rápido')

## 5. Esqueleto de FastAPI con PydanticAI

In [ ]:
# Este código muestra la estructura de un endpoint FastAPI con PydanticAI
# No ejecutar directamente en Jupyter — usar con uvicorn

CODIGO_FASTAPI = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel as PBase
from pydantic_ai import Agent
from fastapi.responses import StreamingResponse
import httpx, os

app = FastAPI(title="API Análisis Contratos")

# Agente con structured output
class AnalisisOutput(PBase):
    riesgo: str
    recomendacion: str
    aprobado: bool

agente = Agent(
    model="anthropic:claude-haiku-4-5-20251001",
    result_type=AnalisisOutput,
    system_prompt="Analiza contratos e identifica riesgos.",
)

class SolicitudAnalisis(PBase):
    texto: str

@app.post("/analizar", response_model=AnalisisOutput)
async def analizar(solicitud: SolicitudAnalisis):
    try:
        result = await agente.run(solicitud.texto)
        return result.data
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/analizar/stream")
async def analizar_stream(solicitud: SolicitudAnalisis):
    agente_txt = Agent(model="anthropic:claude-haiku-4-5-20251001")

    async def generar():
        async with agente_txt.run_stream(solicitud.texto) as stream:
            async for delta in stream.stream_text(delta=True):
                yield delta

    return StreamingResponse(generar(), media_type="text/plain")

# Arrancar con: uvicorn main:app --reload
'''

print('Estructura de FastAPI con PydanticAI:')
print(CODIGO_FASTAPI)
print('\nGuarda este código en main.py y ejecuta: uvicorn main:app --reload')